# Módulo 9 · Práctica: un MLP de caracteres al estilo Bengio (2003)

En el módulo 1 predijiste la siguiente letra mirando **una** letra y contando. Ahora mirarás **tres** y aprenderás con gradientes: embeddings, capa oculta y softmax (Bengio et al., 2003). Todo en NumPy y con **retropropagación a mano**; en el módulo 10 lo hará PyTorch.

| Parte | Qué haces |
| --- | --- |
| 0 | Datos, vocabulario y la línea base de bigramas |
| 1 | Construir el conjunto de contextos de 3 letras |
| 2 | Inicialización de He |
| 3 | Pasada hacia delante y entropía cruzada |
| 4 | Retropropagación manual y comprobación numérica |
| 5 | Entrenamiento y comparación de perplejidades |
| 6 | Generar nombres y mirar los embeddings |
| 7 | Retos |

Completa cada `# TU CÓDIGO` y ejecuta la `# Comprobación` siguiente. Ejecución total: < 1 min en CPU.

## Parte 0 · Datos y línea base

Un solo hilo de BLAS: con matrices tan pequeñas, varios hilos van *más lentos*.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
nombres = [
    'alejandro', 'antonio', 'manuel', 'jose', 'francisco', 'david', 'juan', 'javier', 'daniel', 'carlos',
    'jesus', 'miguel', 'rafael', 'pablo', 'pedro', 'angel', 'sergio', 'fernando', 'jorge', 'luis',
    'alberto', 'alvaro', 'adrian', 'diego', 'raul', 'enrique', 'ramon', 'ivan', 'ruben', 'oscar',
    'andres', 'vicente', 'joaquin', 'santiago', 'victor', 'eduardo', 'mario', 'roberto', 'jaime', 'ignacio',
    'marcos', 'alfonso', 'jordi', 'hugo', 'ricardo', 'salvador', 'guillermo', 'gabriel', 'emilio', 'gonzalo',
    'julio', 'julian', 'martin', 'tomas', 'marc', 'agustin', 'nicolas', 'samuel', 'felix', 'ismael',
    'cristian', 'aitor', 'joan', 'josep', 'lucas', 'mateo', 'leo', 'bruno', 'enzo', 'izan',
    'alex', 'gael', 'thiago', 'marco', 'dario', 'unai', 'iker', 'eric', 'rodrigo', 'aaron',
    'nil', 'pau', 'oliver', 'liam', 'adam', 'biel', 'mauro', 'saul', 'cesar', 'lorenzo',
    'benito', 'esteban', 'felipe', 'gregorio', 'hector', 'horacio', 'jacinto', 'leandro', 'lisandro', 'matias',
    'nestor', 'octavio', 'patricio', 'ramiro', 'sebastian', 'teodoro', 'ulises', 'valentin', 'xavier', 'yago',
    'zacarias', 'aurelio', 'baltasar', 'ceferino', 'domingo', 'elias', 'fabian', 'gaspar', 'hilario', 'iñigo',
    'nuño', 'toño', 'maria', 'carmen', 'ana', 'isabel', 'laura', 'cristina', 'marta', 'lucia',
    'paula', 'elena', 'sara', 'raquel', 'rosa', 'pilar', 'manuela', 'concepcion', 'mercedes', 'julia',
    'beatriz', 'nuria', 'silvia', 'irene', 'patricia', 'rocio', 'andrea', 'montserrat', 'monica', 'alba',
    'sonia', 'sandra', 'marina', 'angela', 'sofia', 'alicia', 'susana', 'yolanda', 'claudia', 'natalia',
    'margarita', 'eva', 'inmaculada', 'noelia', 'veronica', 'esther', 'carla', 'nerea', 'daniela', 'alejandra',
    'martina', 'valeria', 'chloe', 'emma', 'olivia', 'jimena', 'vega', 'lola', 'carlota', 'mia',
    'valentina', 'aitana', 'noa', 'alma', 'triana', 'candela', 'ainhoa', 'ines', 'adriana', 'aroa',
    'blanca', 'celia', 'diana', 'elsa', 'fatima', 'gloria', 'helena', 'iria', 'lara', 'lidia',
    'lorena', 'luna', 'maite', 'mar', 'miriam', 'nadia', 'nieves', 'noemi', 'olga', 'paloma',
    'regina', 'rebeca', 'salma', 'tamara', 'teresa', 'ursula', 'victoria', 'ximena', 'zoe', 'aurora',
    'begoña', 'catalina', 'dolores', 'encarnacion', 'francisca', 'gema', 'juana', 'leire', 'macarena', 'nayara',
    'oihane', 'piedad', 'remedios', 'soledad', 'trinidad', 'vanesa', 'yaiza', 'amparo', 'araceli', 'asuncion',
    'cecilia', 'consuelo', 'emilia', 'esperanza', 'estefania', 'eugenia', 'guadalupe', 'josefa', 'lourdes', 'maribel',
    'purificacion', 'rosario', 'virginia', 'almudena', 'ariadna', 'berta', 'carolina', 'debora', 'estela', 'fernanda',
    'graciela', 'jazmin', 'magdalena', 'ofelia', 'pepa', 'rut', 'sabina', 'tatiana', 'yasmina', 'zaira',
    'ainara', 'arantxa', 'iratxe', 'naia', 'uxue', 'ane', 'june', 'maialen', 'itziar', 'alonso',
    'ernesto', 'marcelo', 'bernardo', 'cayetano', 'rosalia', 'anabel', 'candido', 'clemente', 'efren', 'genaro',
    'isidro', 'leonardo', 'maximo', 'norberto', 'rogelio', 'silvestre', 'tadeo', 'amelia', 'bianca', 'cintia',
    'dafne', 'erika', 'flor', 'ivonne', 'judith', 'karen', 'leticia', 'micaela', 'nora', 'perla',
    'rita', 'selena', 'telma', 'vera', 'wendy', 'abril',
]
print(len(nombres), 'nombres')

In [ ]:
vocab = list(".abcdefghijklmnñopqrstuvwxyz")
V = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for i, c in enumerate(vocab)}

# Misma partición que en el módulo 1
rng = np.random.default_rng(7)
idx = rng.permutation(len(nombres))
n_val = round(0.15 * len(nombres))
val = [nombres[i] for i in idx[:n_val]]
train = [nombres[i] for i in idx[n_val:]]
print(V, "tokens ·", len(train), "entrenamiento ·", len(val), "validación")

**Línea base:** el bigrama suavizado del módulo 1, con α elegido en validación.

In [ ]:
def tokenizar(p):
    return [stoi[c] for c in "." + p + "."]

N = np.zeros((V, V))
for p in train:
    t = tokenizar(p)
    for a, b in zip(t, t[1:]):
        N[a, b] += 1

def ppl_bigrama(alpha, palabras):
    P = (N + alpha) / (N + alpha).sum(axis=1, keepdims=True)
    ll = [np.log(P[a, b]) for p in palabras for a, b in zip(tokenizar(p), tokenizar(p)[1:])]
    return np.exp(-np.mean(ll))

alphas = [0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
ppls = [ppl_bigrama(a, val) for a in alphas]
alpha_opt = alphas[int(np.argmin(ppls))]
PPL_BIGRAMA = min(ppls)
print(f"Mejor alpha = {alpha_opt}: PPL validación del bigrama = {PPL_BIGRAMA:.2f}  (uniforme: {V})")

## Parte 1 · Contextos de 3 letras

Cada nombre genera un ejemplo por letra (más el `.` final). El contexto son las 3 letras anteriores, rellenando con `.` al principio:

| contexto | siguiente |
| --- | --- |
| `. . .` | a |
| `. . a` | n |
| `. a n` | a |
| `a n a` | . |

**Ejercicio 1.** Completa `construir_dataset(palabras, B)`: devuelve `X` (enteros, forma n × B) e `Y` (forma n).

In [ ]:
B = 3   # longitud del contexto

def construir_dataset(palabras, B=3):
    X, Y = [], []
    for p in palabras:
        ctx = [0] * B
        for c in p + ".":
            # TU CÓDIGO: añade ctx a X, el índice de c a Y, y desplaza ctx
            pass
    return np.array(X, dtype=int).reshape(-1, B), np.array(Y, dtype=int)

Xtr, Ytr = construir_dataset(train, B)
Xva, Yva = construir_dataset(val, B)
print(Xtr.shape, Ytr.shape, Xva.shape)

In [ ]:
# Comprobación
Xa, Ya = construir_dataset(["ana"], 3)
assert Xa.tolist() == [[0,0,0],[0,0,1],[0,1,14],[1,14,1]], f"Contextos de 'ana' incorrectos: {Xa.tolist()}"
assert Ya.tolist() == [1, 14, 1, 0], "Objetivos de 'ana' incorrectos"
assert len(Xtr) == sum(len(p) + 1 for p in train), "Cada nombre de longitud L aporta L + 1 ejemplos"
print("Correcto")

## Parte 2 · Parámetros e inicialización

Arquitectura (d = 10, H = 64):

$$E = C[X] \in \mathbb{R}^{n\times 3d},\quad Z = E W_1 + b_1,\quad A = \mathrm{ReLU}(Z),\quad \text{logits} = A W_2 + b_2$$

**Ejercicio 2.** Inicializa `W1` con **He** (desviación $\sqrt{2/n_\text{in}}$, con $n_\text{in}=3d$), `b1` y `b2` a cero y `W2` con desviación $0{,}1/\sqrt{H}$ (logits iniciales casi nulos: la red empieza «sin opinión»). `C` ya está: normal estándar.

In [ ]:
d, H = 10, 64

def init_params(seed=0):
    r = np.random.default_rng(seed)
    C = r.normal(0, 1, (V, d))
    # TU CÓDIGO
    W1 = None
    b1 = None
    W2 = None
    b2 = None
    return {"C": C, "W1": W1, "b1": b1, "W2": W2, "b2": b2}

params = init_params()
print("Parámetros:", sum(v.size for v in params.values()), "  (bigrama:", V * V, ")")

In [ ]:
# Comprobación
assert params["W1"].shape == (B * d, H) and params["W2"].shape == (H, V), "Formas de W1 o W2 incorrectas"
assert params["b1"].shape == (H,) and np.all(params["b1"] == 0) and np.all(params["b2"] == 0), "Sesgos a cero"
assert abs(params["W1"].std() - np.sqrt(2 / (B * d))) < 0.03, "W1 no sigue la inicialización de He"
assert abs(params["W2"].std() - 0.1 / np.sqrt(H)) < 0.003, "Desviación de W2 incorrecta"
assert sum(v.size for v in params.values()) == 4084
print("Correcto")

## Parte 3 · Hacia delante

**Ejercicio 3a.** `softmax_ce(logits, Y)` devuelve la pérdida media y su gradiente respecto a los logits:

$$\mathcal{L} = -\frac{1}{n}\sum_i \log p_{i,y_i},\qquad \frac{\partial \mathcal{L}}{\partial \text{logits}} = \frac{1}{n}\,(P - \text{onehot}(Y))$$

Hazla estable restando el máximo de cada fila y calcula $\log p$ como $z_y - \log\sum_j e^{z_j}$ (log-sum-exp) para no hacer nunca `log(0)`.

In [ ]:
def softmax_ce(logits, Y):
    n = len(Y)
    # TU CÓDIGO
    loss, dlogits = None, None
    return loss, dlogits

In [ ]:
# Comprobación con el ejemplo resuelto del documento (sección 9.6): z = (2, 1, -2), clase correcta 1
L_ej, dz_ej = softmax_ce(np.array([[2., 1., -2.]]), np.array([1]))
assert abs(L_ej - 1.3266) < 1e-3, f"Pérdida {L_ej}, esperaba 1,3266"
assert np.allclose(dz_ej, [[0.7214, -0.7346, 0.0132]], atol=1e-4), f"Gradiente {dz_ej}"
assert np.isfinite(softmax_ce(np.array([[1000., 0., -1000.]]), np.array([2]))[0]), "No es estable con logits grandes"
print("Correcto")

**Ejercicio 3b.** `forward(params, X)` devuelve los logits y una caché con lo que necesitará la retropropagación.

In [ ]:
def forward(params, X):
    C, W1, b1, W2, b2 = (params[k] for k in ("C", "W1", "b1", "W2", "b2"))
    # TU CÓDIGO: E (n, 3d), Z, A, logits
    E = Z = A = logits = None
    cache = (X, E, Z, A)
    return logits, cache

logits, _ = forward(params, Xtr)
print("Pérdida inicial:", softmax_ce(logits, Ytr)[0], "  ln V =", np.log(V))

In [ ]:
# Comprobación
assert logits.shape == (len(Xtr), V), "Los logits deben tener forma (n, V)"
L0 = softmax_ce(logits, Ytr)[0]
assert abs(L0 - np.log(V)) < 0.1, f"Con logits casi nulos la pérdida debe rondar ln V = {np.log(V):.3f}; sale {L0:.3f}"
print("Correcto: la red empieza como un modelo uniforme")

## Parte 4 · Retropropagación a mano

Sigue el grafo hacia atrás (sección 9.6 del documento). Con $G = \partial\mathcal{L}/\partial\text{logits}$:

$$\nabla W_2 = A^\top G,\quad \nabla b_2 = \textstyle\sum_i G_i,\quad \nabla A = G W_2^\top,\quad \nabla Z = \nabla A \odot \mathbb{1}[Z>0]$$
$$\nabla W_1 = E^\top \nabla Z,\quad \nabla b_1 = \textstyle\sum_i \nabla Z_i,\quad \nabla E = \nabla Z\, W_1^\top$$

Para `C`: cada fila de $\nabla E$ se reparte en 3 trozos de tamaño d y se **suma** en la fila de `C` del carácter correspondiente. Usa `np.add.at(dC, X, dE.reshape(n, B, d))` (con `dC[X] += …` se perderían las repeticiones).

**Ejercicio 4.** Completa `backward`.

In [ ]:
def backward(params, cache, dlogits):
    X, E, Z, A = cache
    n = len(X)
    # TU CÓDIGO
    grads = {"C": None, "W1": None, "b1": None, "W2": None, "b2": None}
    return grads

In [ ]:
# Comprobación: gradiente numérico por diferencias centradas (módulo 4)
def perdida_total(p, X, Y):
    return softmax_ce(forward(p, X)[0], Y)[0]

pc = init_params(seed=3)
pc["W2"] = pc["W2"] * 30           # logits no triviales para que el test sea exigente
pc["b1"] = pc["b1"] + 0.1
Xc, Yc = Xtr[:50], Ytr[:50]
lg, cache = forward(pc, Xc)
g = backward(pc, cache, softmax_ce(lg, Yc)[1])
r = np.random.default_rng(0)
for k in pc:
    for _ in range(5):
        i = tuple(r.integers(0, s) for s in pc[k].shape)
        if k == "C":
            i = (int(r.choice(Xc.ravel())), i[1])   # una fila de C que sí se usa
        old = pc[k][i]; h = 1e-5
        pc[k][i] = old + h; Lp = perdida_total(pc, Xc, Yc)
        pc[k][i] = old - h; Lm = perdida_total(pc, Xc, Yc)
        pc[k][i] = old
        num = (Lp - Lm) / (2 * h)
        assert abs(num - g[k][i]) < 1e-6 + 1e-4 * abs(num), f"Gradiente de {k}{i}: analítico {g[k][i]:.6g}, numérico {num:.6g}"
print("Correcto: los 25 gradientes analíticos coinciden con los numéricos")

## Parte 5 · Entrenamiento

Con 1.885 ejemplos cabe todo en un lote: usamos descenso de gradiente por lotes completos con η = 0,5. El corpus es diminuto y la red tiene 4.084 parámetros, así que sobreajusta: añadimos **decaimiento de pesos** (módulo 7), que suma λ·θ a cada gradiente, con λ = 0,005.

**Ejercicio 5.** Escribe el paso de actualización dentro del bucle.

In [ ]:
def entrenar(pasos=1000, lr=0.5, wd=5e-3, seed=0, cada=50):
    p = init_params(seed)
    hist = []
    for paso in range(pasos):
        logits, cache = forward(p, Xtr)
        loss, dlog = softmax_ce(logits, Ytr)
        grads = backward(p, cache, dlog)
        # TU CÓDIGO: para cada k, p[k] -= lr * (grads[k] + wd * p[k])
        if paso % cada == 0 or paso == pasos - 1:
            lv = softmax_ce(forward(p, Xva)[0], Yva)[0]
            hist.append((paso, loss, lv))
    return p, hist

t0 = time.time()
params, hist = entrenar()
print(f"Entrenado en {time.time() - t0:.1f} s")
pasos_h, ltr, lva = map(np.array, zip(*hist))
plt.plot(pasos_h, np.exp(ltr), label="entrenamiento")
plt.plot(pasos_h, np.exp(lva), label="validación")
plt.axhline(PPL_BIGRAMA, ls="--", c="gray", label="bigrama (validación)")
plt.yscale("log"); plt.xlabel("paso"); plt.ylabel("perplejidad"); plt.legend(); plt.show()

In [ ]:
# Comprobación
ppl_tr = np.exp(softmax_ce(forward(params, Xtr)[0], Ytr)[0])
ppl_va = np.exp(softmax_ce(forward(params, Xva)[0], Yva)[0])
print(f"MLP: PPL entrenamiento {ppl_tr:.2f} · validación {ppl_va:.2f}   |   bigrama validación {PPL_BIGRAMA:.2f}")
assert ppl_tr < 8, "La pérdida de entrenamiento no ha bajado lo suficiente: revisa la actualización"
assert ppl_va < PPL_BIGRAMA, "El MLP debería superar al bigrama en validación"
print(f"Correcto: el MLP reduce la perplejidad de validación un {100 * (1 - ppl_va / PPL_BIGRAMA):.0f} %")

**Preguntas.** (1) La PPL de entrenamiento queda muy por debajo de la de validación: ¿qué indica? Prueba `wd=0`. (2) Una tabla de cuatrigramas tendría 28⁴ = 614.656 casillas para 1.885 ejemplos. ¿Por qué el MLP, con 4.084 parámetros, sí aprovecha ese contexto?

## Parte 6 · Generar y mirar dentro

**Ejercicio 6.** Completa `generar`: calcula la distribución con `forward` sobre el contexto actual, aplica la temperatura, sortea y desplaza el contexto.

In [ ]:
def generar(params, rng, T=1.0, max_len=20):
    ctx = [0] * B
    out = ""
    for _ in range(max_len):
        logits, _ = forward(params, np.array([ctx]))
        # TU CÓDIGO: probabilidades con temperatura T, sortear j, parar si j == 0
        break
    return out

g_rng = np.random.default_rng(42)
print([generar(params, g_rng) for _ in range(15)])

In [ ]:
# Comprobación
muestras = [generar(params, np.random.default_rng(s)) for s in range(200)]
assert all(set(m) <= set(vocab[1:]) for m in muestras), "Los nombres no deben contener '.'"
assert np.mean([len(m) for m in muestras]) > 3, "Los nombres generados son demasiado cortos"
nuevos = sum(m not in nombres for m in muestras)
print(f"Correcto. {nuevos} de 200 nombres no estaban en el corpus")

In [ ]:
# Embeddings aprendidos proyectados en 2D con PCA (módulo 3)
Cc = params["C"] - params["C"].mean(axis=0)
U, S, Vt = np.linalg.svd(Cc, full_matrices=False)
P2 = Cc @ Vt[:2].T
plt.figure(figsize=(6, 6))
for i, c in enumerate(vocab):
    plt.scatter(*P2[i], c="tab:red" if c in "aeiou" else "tab:blue")
    plt.annotate(c, P2[i], fontsize=13)
plt.title("Embeddings de caracteres (rojo: vocales)"); plt.show()

**Pregunta.** ¿Se agrupan las vocales? Nadie le dijo a la red qué es una vocal: aparecen en contextos parecidos. Así funcionan los embeddings de un LLM.

## Parte 7 · Retos

1. **Test.** Separa también un conjunto de *test*: elige α y λ en validación y compara en *test*.
2. **GELU.** Sustituye ReLU por GELU en `forward` y `backward` y repite la comprobación numérica.
3. **SwiGLU.** Usa $A = \mathrm{SiLU}(EW_g)\odot(EW_u)$ con H = 43 (mismos parámetros) y deriva su retropropagación.
4. **Contexto.** Prueba B = 2, 4, 5. ¿Mejora siempre la validación?
5. **Inicialización.** Pon `W2` con desviación 1. ¿Qué pasa con la pérdida inicial?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1 (dentro del bucle)
#             X.append(ctx); Y.append(stoi[c]); ctx = ctx[1:] + [stoi[c]]
# Ejercicio 2
#     W1 = r.normal(0, np.sqrt(2 / (B * d)), (B * d, H))
#     b1 = np.zeros(H)
#     W2 = r.normal(0, 0.1 / np.sqrt(H), (H, V))
#     b2 = np.zeros(V)
# Ejercicio 3a
#     z = logits - logits.max(axis=1, keepdims=True)
#     P = np.exp(z); P /= P.sum(axis=1, keepdims=True)
#     loss = -(z[np.arange(n), Y] - np.log(np.exp(z).sum(axis=1))).mean()
#     dlogits = P.copy(); dlogits[np.arange(n), Y] -= 1; dlogits /= n
# Ejercicio 3b
#     E = C[X].reshape(len(X), -1)
#     Z = E @ W1 + b1
#     A = np.maximum(Z, 0)
#     logits = A @ W2 + b2
# Ejercicio 4
#     dW2 = A.T @ dlogits;  db2 = dlogits.sum(axis=0)
#     dA = dlogits @ params["W2"].T
#     dZ = dA * (Z > 0)
#     dW1 = E.T @ dZ;  db1 = dZ.sum(axis=0)
#     dE = dZ @ params["W1"].T
#     dC = np.zeros_like(params["C"]); np.add.at(dC, X, dE.reshape(n, B, d))
#     grads = {"C": dC, "W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
# Ejercicio 5 (dentro del bucle)
#         for k in p:
#             p[k] -= lr * (grads[k] + wd * p[k])
# Ejercicio 6 (dentro del bucle)
#         z = logits[0] / T
#         pr = np.exp(z - z.max()); pr /= pr.sum()
#         j = rng.choice(V, p=pr)
#         if j == 0:
#             break
#         out += itos[j]; ctx = ctx[1:] + [j]